# M3 — DEA: Análise Envoltória de Dados

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Caso: Qual hospital municipal é mais eficiente?

A Secretaria Estadual de Saúde do PR contratou a Gradus para identificar quais dos 5 hospitais municipais de médio porte estão sendo mais eficientes em transformar recursos em serviços.

**Inputs (3):** orçamento anual (R\$ M), leitos, médicos FTE.  
**Outputs (2):** atendimentos/ano, altas/ano.

Não podemos usar uma simples razão (output/input) porque há múltiplos inputs e múltiplos outputs.

**DEA** resolve isso: para cada DMU $k$, monta um PL que tenta combinar os outros DMUs para produzir o que $k$ produz com **menos inputs**. Se conseguir, $k$ é ineficiente.

> ⚠️ **Versão TEMPLATE** — algumas células contêm `# TODO` para você preencher. Se ficar travado, abra a versão solução: `m3_dea_solution.ipynb` (ou o `.py` em `scripts/`). Se estiver no Colab, o link da solução está no deck.

## Setup — dois solvers lado a lado

OR-Tools (open-source) + Gurobi (comercial, Genoa representa no Brasil). DEA é puramente LP — ambos resolvem trivialmente, e o ponto interessante é **quando você roda 100s de DMUs em sequência, a velocidade do Gurobi compensa**.

In [ ]:
%pip install -q ortools gurobipy pandas

In [ ]:
from ortools.linear_solver import pywraplp
import pandas as pd
import numpy as np

# Os 5 hospitais municipais
df = pd.DataFrame({
    'hospital': ['Maringá', 'Cascavel', 'Londrina', 'Ponta Grossa', 'Foz do Iguaçu'],
    'orc':     [18, 22, 28, 16, 24],     # R$ M
    'leitos':  [120, 140, 180, 100, 150],
    'medicos': [80, 95, 130, 70, 110],
    'atend':   [95, 100, 145, 85, 105],  # mil/ano
    'altas':   [12, 13, 18, 10, 14],     # mil/ano
})

INPUTS  = ['orc', 'leitos', 'medicos']
OUTPUTS = ['atend', 'altas']
df

## Modelo CCR-input-oriented

Para cada hospital $k$ (DMU em avaliação), resolvemos:

$$\min\ \theta_k \quad \text{s.a.}$$

- Para cada input $i$: $\sum_j \lambda_j\, x_{ij} \le \theta_k\, x_{ik}$
- Para cada output $r$: $\sum_j \lambda_j\, y_{rj} \ge y_{rk}$
- $\lambda_j \ge 0$

Interpretação:
- $\theta_k = 1$: hospital $k$ está na fronteira (eficiente)
- $\theta_k < 1$: ineficiente — poderia produzir o mesmo com $\theta_k$ dos inputs atuais
- $\lambda_j > 0$: hospital $j$ é peer (benchmark) de $k$

In [ ]:
def dea_ccr_input(df, inputs, outputs):
    """DEA CCR input-oriented: resolve um PL por DMU.
    Retorna dict com theta_k para cada DMU k."""

    from ortools.linear_solver import pywraplp
    DMU = df.index.tolist()
    n, m, q = len(DMU), len(inputs), len(outputs)
    X = df[inputs].values    # n x m
    Y = df[outputs].values   # n x q

    resultados = {}
    for k in range(n):
        s = pywraplp.Solver.CreateSolver('GLOP')

        # TODO: variaveis
        # theta_k em [0, 1]
        # lambda_j >= 0 para cada DMU j

        # TODO: restricoes
        # (inputs) sum_j lambda[j] * X[j,i] <= theta_k * X[k,i] forall i
        # (outputs) sum_j lambda[j] * Y[j,r] >= Y[k,r] forall r

        # TODO: FO + solve
        # s.Minimize(theta_k); s.Solve()
        # resultados[DMU[k]] = theta_k.solution_value()

    raise NotImplementedError("Complete o DEA CCR")
    return resultados


### O mesmo DEA em Gurobi

Mesma estrutura — 1 LP por DMU, em loop. Quando a amostra cresce (centenas de agências bancárias, milhares de lojas), o tempo agregado de todos os LPs vira mensurável; Gurobi resolve cada um em poucos ms.

Vamos também comparar tempos OR-Tools (GLOP) vs Gurobi nos 15 hospitais.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def dea_ccr_input_gurobi(df, inputs, outputs):
    """Mesmo DEA CCR, em Gurobi. Note que aqui eh LP puro (sem binarias)."""
    # TODO: itere por DMU, monte o LP, resolva via m.optimize()
    # Use m.addVar para theta, m.addVars para lambda; m.addConstrs com generator

    raise NotImplementedError("Complete o DEA Gurobi")


### Discussão

- **Maringá, Londrina e Ponta Grossa** estão na fronteira ($\theta = 1$).
- **Cascavel** ($\theta = 0,929$) poderia produzir os mesmos outputs com 92,9 % dos inputs — ~7 % de "folga" relativa. Seu peer é Maringá com $\lambda = 1,083$.
- **Foz do Iguaçu** ($\theta = 0,933$) idem — Maringá também é seu peer.

Maringá é o **benchmark dominante** — os ineficientes deveriam estudar como Maringá organiza recursos.

**Atenção:** DEA é *comparativo*. "Eficiente" aqui não significa "melhor possível em teoria" — só significa que nenhum outro DMU desta amostra produz mais com menos. Se você tirar Maringá, a fronteira muda.

---

## Exercício de extensão: 15 hospitais + BCC + super-eficiência

Vamos ampliar a amostra para 15 hospitais (os 5 originais + 10 sintéticos) e introduzir três técnicas que aparecem no uso prático de DEA:

### 1. Modelo BCC (retornos variáveis de escala)

Diferença vs CCR: adiciona a restrição $\sum_j \lambda_j = 1$. Significa que a combinação linear dos peers deve ter "tamanho" similar ao avaliado.

- **CCR** assume retornos *constantes* de escala — dobrar inputs deveria dobrar outputs.
- **BCC** permite retornos *variáveis* — eficiência pura, separando do efeito escala.

$$\text{Eficiência de escala} = \frac{\theta_{CCR}}{\theta_{BCC}}$$

Se BCC = 1 mas CCR < 1, o hospital é tecnicamente eficiente mas está no tamanho "errado" (escala subótima).

In [ ]:
# 15 hospitais (5 originais + 10 sinteticos)

# TODO: implemente o modelo BCC (VRS):
#   mesmo CCR + 1 restricao adicional: sum_j lambda[j] == 1
# Compare theta_CCR vs theta_BCC. A razao theta_CCR / theta_BCC eh a eficiencia de escala.

raise NotImplementedError("Complete o BCC")


### Leitura do CCR vs BCC

Observe que **CCR é sempre mais restritivo** que BCC (resolve um problema com menos liberdade), então θ_CCR ≤ θ_BCC para qualquer DMU.

- DMU com **CCR < BCC = 1**: tecnicamente eficiente, mas problema de escala (está pequeno ou grande demais).
- DMU com **CCR = BCC = 1**: eficiência total — operando na escala ótima.
- DMU com **CCR = BCC < 1**: ineficiente puramente por *má utilização* de recursos, não por escala.

### 2. Análise de slack (TODO)

Mesmo com θ = 1, pode sobrar folga em alguma dimensão. Para detectar, rode uma **segunda etapa**: dado θ ótimo, maximize a soma dos slacks. Se > 0, a DMU é "pseudo-eficiente".

In [ ]:
# TODO: Analise de slack (2a fase)
# Para DMUs com theta = 1, fixe theta e maximize sum dos slacks:
#   max sum_i s_i^- + sum_r s_r^+
#   s.a. sum_j lambda[j]*X[j,i] + s_i^- == theta_k*X[k,i]
#        sum_j lambda[j]*Y[j,r] - s_r^+ == Y[k,r]
# Se algum s > 0, a DMU eh eficiente fraca (nao Pareto).

raise NotImplementedError("Complete a analise de slack")


### 3. Super-eficiência (Andersen-Petersen)

Para *rankear* os eficientes (todos têm θ = 1, então CCR/BCC empata), rodamos o modelo **excluindo cada DMU do seu próprio conjunto de referência**:

$$\min\ \theta_k^{AP} \quad \text{s.a.}\quad \sum_{j \ne k} \lambda_j x_{ij} \le \theta_k x_{ik},\ \sum_{j \ne k} \lambda_j y_{rj} \ge y_{rk}$$

Agora $\theta_k^{AP}$ pode ser > 1 (interpretação: quanto $k$ é "mais eficiente" que os demais). Ordena a fronteira.

In [ ]:
def super_eficiencia(df, inputs, outputs):
    """Andersen-Petersen 1993: para cada DMU k, remove-a do conjunto de referencia.
    Se theta_k > 1, ela eh super-eficiente."""

    # TODO: itere por DMU k.
    # Para cada k, monte o CCR mas com soma `for j in range(n) if j != k`.
    # Aceite theta_k sem upper bound de 1 (pode passar de 1).
    # Detecte inviabilidade e marque como "extreme efficient".

    raise NotImplementedError("Complete a super-eficiencia")


### Para discutir

1. **CCR vs BCC nos 5 originais:** quais hospitais têm ineficiência de escala?
2. **Peer analysis na amostra grande:** quais hospitais são peers "dominantes" (aparecem como benchmark para muitos)?
3. **Super-eficiência:** dentre os hospitais eficientes pelo CCR, qual é o mais "robusto" (maior $\theta_{AP}$)?
4. **Quando usar DEA na consultoria:** lista de situações típicas — agências bancárias, lojas de varejo, unidades escolares, ESFs, distribuidoras. O que precisa ser homogêneo?